In [17]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

def afficher_planche_contact(dossier_classe, nb_colonnes=8, nb_lignes=8):
    fichiers = sorted(os.listdir(dossier_classe))
    figure, axes = plt.subplots(nb_lignes, nb_colonnes, figsize=(16, 16))

    index = 0
    for ligne in range(nb_lignes):
        for colonne in range(nb_colonnes):
            axe = axes[ligne][colonne]
            if index < len(fichiers):
                chemin = os.path.join(dossier_classe, fichiers[index])
                try:
                    image = Image.open(chemin)
                    axe.imshow(image)
                    axe.set_title(fichiers[index], fontsize=6)
                except Exception:
                    pass
            axe.axis("off")
            index += 1

    plt.tight_layout()
    plt.show()

# exemple : afficher_planche_contact("atelier_prepa_donnees_images/data/raw/glass")

# partie 1-Exploration du dataset

In [15]:


dossier_raw = "../data/raw"

def explorer_dataset(dossier_raw):
    lignes = []
    classes = sorted(os.listdir(dossier_raw))

    for classe in classes:
        dossier_classe = os.path.join(dossier_raw, classe)
        noms_fichiers = sorted(os.listdir(dossier_classe))

        for nom_fichier in noms_fichiers:
            chemin = os.path.join(dossier_classe, nom_fichier)
            taille_octets = os.path.getsize(chemin)

            ligne = {
                "nom": nom_fichier,
                "classe": classe,
                "chemin": chemin,
                "format": None,
                "mode": None,
                "largeur": None,
                "hauteur": None,
                "ecart_type": None,
                "nb_canaux": None,
                "taille_octets": taille_octets,
                "corrompue": False
            }

            try:
                image = Image.open(chemin)
                image.load()

                info_format = image.format
                info_mode = image.mode
                info_largeur = image.width
                info_hauteur = image.height

                if info_mode == "L":
                    nb_canaux = 1
                elif info_mode == "RGB":
                    nb_canaux = 3
                elif info_mode == "RGBA":
                    nb_canaux = 4
                else:
                    nb_canaux = len(image.getbands())

                tableau = np.array(image)
                ecart_type = round(float(tableau.std()), 2)

                ligne["format"] = info_format
                ligne["mode"] = info_mode
                ligne["largeur"] = info_largeur
                ligne["hauteur"] = info_hauteur
                ligne["nb_canaux"] = nb_canaux
                ligne["ecart_type"] = ecart_type

            except Exception as erreur:
                ligne["corrompue"] = True

            lignes.append(ligne)

    return lignes


lignes = explorer_dataset(dossier_raw)
print("Nombre total d'images explorees :", len(lignes))
print("Cles disponibles :", lignes[0].keys())

Nombre total d'images explorees : 1032
Cles disponibles : dict_keys(['nom', 'classe', 'chemin', 'format', 'mode', 'largeur', 'hauteur', 'ecart_type', 'nb_canaux', 'taille_octets', 'corrompue'])


# Partie 2 – Détecter les images corrompues

In [3]:
def est_corrompue(chemin_image):
    """Retourne True si l'image ne peut pas etre ouverte/lue correctement."""
    try:
        image = Image.open(chemin_image)
        image.load()
        return False
    except Exception:
        return True


dossier_raw = "../data/raw"
classes = sorted(os.listdir(dossier_raw))
images_corrompues = []

for classe in classes:
    dossier_classe = os.path.join(dossier_raw, classe)
    for nom_fichier in sorted(os.listdir(dossier_classe)):
        chemin = os.path.join(dossier_classe, nom_fichier)
        if est_corrompue(chemin):
            images_corrompues.append(chemin)

print("Nombre d'images corrompues :", len(images_corrompues))

Nombre d'images corrompues : 6


# Partie 3 – Détecter les images vides

In [16]:
def est_vide(chemin_image, seuil_ecart_type=5.0):
    """
    Retourne True si l'image est consideree comme vide :
    entierement noire, entierement blanche, ou tres peu de variation.
    """
    try:
        image = Image.open(chemin_image)
        image.load()
    except Exception:
        return False   # une image corrompue est traitee a part (Partie 2)

    tableau = np.array(image)
    moyenne = tableau.mean()
    ecart_type = tableau.std()

    if ecart_type < seuil_ecart_type:
        return True
    if moyenne < 2:
        return True
    if moyenne > 253:
        return True
    return False


# on enregistre le resultat DANS la liste "lignes" pour le reutiliser plus tard
for ligne in lignes:
    if ligne["corrompue"]:
        ligne["vide"] = False
    else:
        ligne["vide"] = est_vide(ligne["chemin"])

images_vides = []
for ligne in lignes:
    if ligne["vide"]:
        images_vides.append(ligne["chemin"])

print("Nombre d'images quasi vides :", len(images_vides))
for chemin in images_vides:
    print("  -", chemin)

Nombre d'images quasi vides : 2
  - ../data/raw\cardboard\image-blanche-512x384.jpg
  - ../data/raw\metal\image-blanche-512x384.jpg


# Partie 4 – Détecter les différences de résolution

1) Déterminer la résolution minimale, la résolution maximale, les résolutions les plus
fréquentes et le nombre d'images par résolution. 
2) On décide qu'une image doit avoir au minimum 64 × 64 pixels. Identifier toutes les images
ne respectant pas cette contrainte.

In [18]:
lignes_valides = []
for ligne in lignes:
    if not ligne["corrompue"]:
        lignes_valides.append(ligne)

print("Nombre d'images valides (hors corrompues) :", len(lignes_valides))
compteur_resolutions = {}
for ligne in lignes_valides:
    resolution = (ligne["largeur"], ligne["hauteur"])
    if resolution in compteur_resolutions:
        compteur_resolutions[resolution] += 1
    else:
        compteur_resolutions[resolution] = 1

toutes_largeurs = []
toutes_hauteurs = []
for ligne in lignes_valides:
    toutes_largeurs.append(ligne["largeur"])
    toutes_hauteurs.append(ligne["hauteur"])

print("Resolution minimale :", min(toutes_largeurs), "x", min(toutes_hauteurs))
print("Resolution maximale :", max(toutes_largeurs), "x", max(toutes_hauteurs))

resolutions_triees = sorted(compteur_resolutions.items(), key=lambda item: item[1], reverse=True)
print("Resolutions les plus frequentes :")
for resolution, nombre in resolutions_triees:
    print("  ", resolution, "->", nombre, "images")

images_trop_petites = []
for ligne in lignes_valides:
    if ligne["largeur"] < 64 or ligne["hauteur"] < 64:
        images_trop_petites.append(ligne["chemin"])

print("Nombre d'images trop petites (< 64x64) :", len(images_trop_petites))
for chemin in images_trop_petites:
    print("  -", chemin)

Nombre d'images valides (hors corrompues) : 1026
Resolution minimale : 32 x 32
Resolution maximale : 512 x 384
Resolutions les plus frequentes :
   (512, 384) -> 1013 images
   (32, 32) -> 5 images
   (48, 32) -> 4 images
   (40, 40) -> 4 images
Nombre d'images trop petites (< 64x64) : 13
  - ../data/raw\cardboard\cardboard117.jpg
  - ../data/raw\cardboard\cardboard22.jpg
  - ../data/raw\cardboard\cardboard70.jpg
  - ../data/raw\glass\glass100.jpg
  - ../data/raw\glass\glass15.jpg
  - ../data/raw\glass\glass21.jpg
  - ../data/raw\glass\glass23.jpg
  - ../data/raw\metal\metal121.jpg
  - ../data/raw\metal\metal2.jpg
  - ../data/raw\metal\metal26.jpg
  - ../data/raw\paper\paper10.jpg
  - ../data/raw\paper\paper54.jpg
  - ../data/raw\paper\paper64.jpg


# Partie 5 – Détecter les différents canaux

In [19]:
lignes_valides = []
for ligne in lignes:
    if not ligne["corrompue"]:
        lignes_valides.append(ligne)

print("Nombre d'images valides (hors corrompues) :", len(lignes_valides))
compteur_canaux = {}
for ligne in lignes_valides:
    nb = ligne["nb_canaux"]
    if nb in compteur_canaux:
        compteur_canaux[nb] += 1
    else:
        compteur_canaux[nb] = 1

print("Repartition par nombre de canaux :", compteur_canaux)

Nombre d'images valides (hors corrompues) : 1026
Repartition par nombre de canaux : {3: 1006, 4: 18, 1: 2}
